# Text Level

Authors: **Josep de Cid** and **Gonzalo Recio**

## Statement

Implement a language identifier for the European languages:
- English, Spanish, Dutch, German, Italian & French.

Use wortschatz leipzig corpora:
- 30k sentences of each of the 6 languages as training set.
- 10k of each as test set.

## Language Identification

For building the language identifier we are going to follow these steps:
- Preprocess:
    - Remove all the digits from data
    - Convert all the texts to lower case
    - Replace continuous white spaces by a single one
    - Concatenate all sentences with a double space in between
- Language Representation: Use character trigrams as language model.
- Statistical Language Model: Use Laplace smoothing technique to avoid zero counts (Remove all trigrams that occurs less than 5 times in training corpus).

In [1]:
import re
import math
import time
import glob
import string
import datetime

import numpy as np

from nltk.tokenize import RegexpTokenizer

### Preprocessing

In [2]:
# Read train data

def read_dataset(mode):
    sentences = {}
    for path in glob.glob(f'data/langId/*_{mode}.txt'):
        lang = path.split('/')[2].split('_')[0]
        with open(path, mode='r', encoding='utf-8') as f:
            sentences[lang] = [l.split('\t')[1].strip() for l in f.readlines()]
    
    return sentences
                          
train_sentences = read_dataset(mode='trn')
test_sentences = read_dataset(mode='tst')

In [3]:
# Preprocess texts for all languages

langs = ['deu', 'eng', 'fra', 'ita', 'nld', 'spa']

processed_sentences = {}

tokenizer = RegexpTokenizer(r'\w+')

def preprocess_sentence(s):
    s = re.sub(r'[0-9]+', '', s).strip()  # remove digits
    s = s.lower()                         # to lowercase
    s = ' '.join(tokenizer.tokenize(s))   # removes double spacing and strange punctuation chars
    return s

for lang in langs:
    processed_sentences[lang] = []
    for s in train_sentences[lang]:
        s = preprocess_sentence(s)
        processed_sentences[lang].append(s)
    
    # now we join sentences with 2 spaces for the trigram language repesentation
    processed_sentences[lang] = '  '.join(processed_sentences[lang])
        
processed_sentences['deu'][:1000]

'wissenschaft welche rolle das licht im wissenschaftsjahr in jena spielt weltweit wird dem thema licht große aufmerksamkeit gewidmet  touristik das magazin für die freiheit unterwegs auto bild launcht neue zeitschrift reisemobil hamburg ots die deutschen sind reiseweltmeister auch auf vier rädern im rollenden appartement  november russland greift nur moderate rebellen an vorwurf des bruchs einer feuerpause im nordwesten  fernsehen münchen ots geplante themen großes theater kleine wirkung  oktober wenn körper geist nicht entspannen können folgende tipps können helfen mit dauerhafter unruhe im alltag umzugehen  min mediathek aufgeheizte stimmung in tröglitz landrat wird mit enthauptung gedroht der landrat ist zuständig für die unterbringung von asylbewerbern im burgenlandkreis  oktober uhr richtige schlüsse ziehen die formel ist nach dem unfall von jules bianchi geschockt  guten morgen mit indus holding krones merck kgaa patrizia immobilien sky deutschland tom tailor inhindus holding wil

### Language model representation

Each language corpus is going to be represented as a set of character trigrams

In [4]:
from nltk.collocations import TrigramCollocationFinder

start = time.time()

language_models = {}

for lang in langs:
    text = processed_sentences[lang]
    print('Modelling language:', lang)
    
    # trigrams generation by chars
    finder = TrigramCollocationFinder.from_words(text)
    # filtering: remove trigrams with less than 5 occurences
    finder.apply_freq_filter(5)
    
    #  FreqDist object
    print('    ', finder.ngram_fd)
    # This object has what we need to compute word occurence probabilities
    print('    Example: counts for trigram "eiw":',finder.ngram_fd[('e','i','w')])
    print('    N:', finder.ngram_fd.N())
    print('    B:', finder.ngram_fd.B())

    # Save language model
    language_models[lang] = finder.ngram_fd
    
end = time.time()
learning_time = end-start
print('\nLearning time:', str(datetime.timedelta(seconds=learning_time)))


Modelling language: deu
     <FreqDist with 8108 samples and 3308796 outcomes>
    Example: counts for trigram "eiw": 46
    N: 3308796
    B: 8108
Modelling language: eng
     <FreqDist with 6259 samples and 3531333 outcomes>
    Example: counts for trigram "eiw": 0
    N: 3531333
    B: 6259
Modelling language: fra
     <FreqDist with 7208 samples and 3590245 outcomes>
    Example: counts for trigram "eiw": 0
    N: 3590245
    B: 7208
Modelling language: ita
     <FreqDist with 5356 samples and 3714685 outcomes>
    Example: counts for trigram "eiw": 0
    N: 3714685
    B: 5356
Modelling language: nld
     <FreqDist with 7590 samples and 3090653 outcomes>
    Example: counts for trigram "eiw": 17
    N: 3090653
    B: 7590
Modelling language: spa
     <FreqDist with 6851 samples and 3853807 outcomes>
    Example: counts for trigram "eiw": 0
    N: 3853807
    B: 6851

Learning time: 0:01:06.658158


### Statistical Model (Laplace smoothing)

In [5]:
# Set a factor for the smoothing
factor = 0.75

# and the probability is computed as P = (counts + factor) / (N + factor*B)

Let's see how to compute it with one Deusche sentence (we have to select the language that obtains a highest probability)

In [6]:
# for this new 'deutsche' sentence these are the scores for each language

text = '26. August 2015 09:41 Anschlag auf Flüchtlingsheim verübt Ein Vermummter warf einen Brandsatz in das noch unbewohntes Gebäude.'
text = preprocess_sentence(text)  # preprocess sentence

log_probs = []

for lang in langs:
    finder = TrigramCollocationFinder.from_words(text)  # convert to trigram character representation
    model = language_models[lang]                       # get model from trai data
    log_prob = 0
    for trigram in finder.ngram_fd.keys():
        P = (model[trigram] + factor) / (model.N() + factor*model.B())
        log_prob += math.log(P) 
    log_probs.append(log_prob)
    print(lang, 'score:', log_prob)

print('\nLanguage identified according to model is:', langs[np.argmax(log_probs)])

deu score: -792.950465647615
eng score: -1002.715867697117
fra score: -1065.2440322022417
ita score: -1133.2685363012172
nld score: -944.993094658438
spa score: -1123.44304373425

Language identified according to model is: deu


Now let's perform this for the whole test dataset

In [7]:
# Read test data and preprocess it

test_sentences = {}

for lang in langs:
    test_sentences[lang] = []
    with open(f'data/langId/{lang}_tst.txt') as f:
        for line in f:
            test_sentences[lang].append(line.split('\t')[1].strip())
test_sentences

{'deu': ['07:00 DGAP-Adhoc: Accu Holding AG: Einladung zur Generalversammlung und Ernennung eines weiteren Mitglieds der Konzernleitung (deutsch) 15.05.',
  '09.11.15 – 01:56 min Mediathek Familiennachzug von Flüchtlingen De-Maizière-Vorstoß entzweit Regierungskoalition Einer dieser Werte ist die Achtung der Familie.',
  '0day-Exploits werden auf dem ohnehin bereits von konkurrierenden Geheimdiensten finanzierten kriminellen Schwarzmarkt für sechs- bis achtstellige Euro-Beträge gehandelt.',
  '1/12 Im Entwerfen von Autos hat Apples iCar-Designer Erfahrung: Marc Newsons Konzeptstudie Ford 021 C, die 1999 in Tokio präsentiert wurde.',
  '11395 Russische Fischer fangen mystischen Fisch mit BrontosauruskopfEin merkwürdiger Fischfang ist ins Netz russischer Fischer im Gebiet Archangelsk geraten, teilte das örtliche Nachrichtenportal 29.ru mit.',
  '11863 Kerry für Sofortverhandlungen mit Russland über IS-Bekämpfung in SyrienDie USA drängen Russland zu Verhandlungen über den Kampf gegen die 

### Computation of tests corpus probabilities and language identification

Now let's compute probabilities for the tests sentences and according to these probabilities we associate a language to the sentence (the one with highest probability).

In [8]:
# Now we compute probabilities and the language identification

start = time.time()

test_results = []

for l in langs:
    sentences = test_sentences[l]
    for s in sentences:
        # for each sentence, we have to compute score for each language
        log_probs = []
        for lang in langs:
            s = preprocess_sentence(s)                       # preprocess sentence
            finder = TrigramCollocationFinder.from_words(s)  # convert to trigram character representation
            model = language_models[lang]                    # get model from train data
            log_prob = 0
            for trigram in finder.ngram_fd.keys():
                P = (model[trigram] + factor) / (model.N() + factor*model.B())
                log_prob += math.log(P)
            log_probs.append(log_prob)
        # we select the language with higher probability (of higher log(prob)).
        identified_lang = langs[np.argmax(log_probs)]
        test_results.append(identified_lang)
    
end = time.time()
test_time = end-start
print('Test time:', str(datetime.timedelta(seconds=test_time)))

Test time: 0:03:19.894794


### Accuracy in NLTK

In [9]:
from nltk.metrics.scores import accuracy

ref   = ['deu']*len(test_sentences['deu']) + \
        ['eng']*len(test_sentences['eng']) + \
        ['fra']*len(test_sentences['fra']) + \
        ['ita']*len(test_sentences['ita']) + \
        ['nld']*len(test_sentences['nld']) + \
        ['spa']*len(test_sentences['spa'])

accuracy = str(accuracy(ref, test_results))
'accuracy: ' + accuracy

'accuracy: 0.9983660403154543'

### Confusion Matrix in NLTK

In [10]:
from nltk.metrics import ConfusionMatrix

cm = ConfusionMatrix(ref, test_results)
print(cm.pretty_format())

    |    d    e    f    i    n    s |
    |    e    n    r    t    l    p |
    |    u    g    a    a    d    a |
----+-------------------------------+
deu |<9979>   5    .    .    5    1 |
eng |    .<9982>   1    1    3    . |
fra |    .   10<9980>   3    3    4 |
ita |    .   13    4<9977>   1    5 |
nld |    6   12    1    3<9976>   2 |
spa |    .    5    1    9    .<9985>|
----+-------------------------------+
(row = reference; col = test)



## Summary of results _(execution times, accuracy and confussion matrix)_

In [11]:
print('Learning time:', str(datetime.timedelta(seconds=learning_time)),' ('+str(learning_time)+' sec.)')
print('Test time:    ', str(datetime.timedelta(seconds=test_time)),' ('+str(test_time)+' sec.)')
print()
print('Accuracy:     ', accuracy)
print()
print('Confusion Matrix:')
print(cm.pretty_format())

Learning time: 0:01:06.658158  (66.65815758705139 sec.)
Test time:     0:03:19.894794  (199.894793510437 sec.)

Accuracy:      0.9983660403154543

Confusion Matrix:
    |    d    e    f    i    n    s |
    |    e    n    r    t    l    p |
    |    u    g    a    a    d    a |
----+-------------------------------+
deu |<9979>   5    .    .    5    1 |
eng |    .<9982>   1    1    3    . |
fra |    .   10<9980>   3    3    4 |
ita |    .   13    4<9977>   1    5 |
nld |    6   12    1    3<9976>   2 |
spa |    .    5    1    9    .<9985>|
----+-------------------------------+
(row = reference; col = test)



# Conclusions

The accuracy is very high, so we can say that the estimator based on observed probabilities for the trigrams from de training data works excellent. This can also be observed from the Confussion Matrix, which shows a diagonal with high values (correct predictions). 

Also, some of the wrongly classified sentense may happen because they contain words from other languages, specially in English. We can observe that many sentences from different languages have been classified as English (2nd column in the confussion matrix).

Examples of 3 italian sentences that we found containing some several English words
> "3186	Farà piacere agli appassionati del 3D laccordo siglato da **The Space Cinema** con **Fifa World Cup South Africa**."

> "8953  Cinema: Il Nuovo **'Fast And Furious'** Ai Nastri Di Partenza - **Yahoo!**"

> "9990	Sono inoltre disponibili i modelli **diesel Mini One D Countryman (66 kW/90 CV)** e **Mini Cooper D Countryman (82 kW/112 CV)**."

> "6180	Politecnico di Torino nel 1982m e Ph.D. alla **University of Waterloo** nel 1986) è un insigne professore di Ingegneria Chimica e Direttore Generale dell **Institute for Chemicals and Fuels from Alternative Resources** alla **University of Western Ontario**."

These can be some of the sentences that are going to be very difficult to classify correctly in any most models. So we can say that the probabilistic estimator we used performs excellent for most of the sentences.


Regarding the execution times, training time is acceptable (1.5 min) bearing in mind that there are a lot of sentenses to process and count trigrams occurences. On the other hand, test time is a bit high (3.5 mins) for being such few less sentences. This is because for each sentence we have to individually convert it into trigrams and then compute the estimator considering trigrams occurences in each of the 6 languages.

As a drawback, observing the test time, we can say that if the amount of sentences was larger and/or the sentences were much longer and/or we had more than 6 languages to classify, this method would become more and more impracticable to compute.